# SBi-FAGAT — Mega-Thread 10k Scanner: Colab & Kaggle GPU Test/Benchmark

Notebook này dùng để:
1. Kiểm thử toàn bộ test suite (`pytest -q`, trong đó có `tests/test_megathread_10k_scanner.py`) và kiểm tra tính toàn vẹn 31 tệp JSON kết quả (`--validate-results`).
2. Đo tốc độ & VRAM thực tế trên GPU **Google Colab (T4 / L4 / A100)** hoặc **Kaggle (T4 / T4 x 2)** cho đường ống **Mega-Thread 10k Scanner** (`stratified` 600 cạnh và `full` 10.000 bình luận).
3. Tùy chọn khởi chạy giao diện **Demo Web MVP** ngay trên Colab.

### Hướng dẫn nhanh
- **Google Colab:** Chọn `Runtime -> Change runtime type -> T4 GPU` (hoặc `L4` / `A100`). Nếu muốn dùng bộ trọng số đã huấn luyện thật, đặt thư mục `checkpoints/` (`dual_ce_vn/` và `best_sbifagat_vn.pt`) tại `MyDrive/SBiFAGAT/checkpoints` trên Google Drive.
- **Kaggle Notebook:** Chọn `Accelerator -> GPU T4 x2` và bật **Internet: ON**. Có thể đính kèm dataset chứa `dual_ce_vn/` và `best_sbifagat_vn.pt` vào `/kaggle/input/...` (không bắt buộc — nếu không gắn checkpoint thật, Cell benchmark sẽ tự động chạy đo thông lượng & VRAM trên `uitnlp/visobert` khởi tạo chuẩn và gắn nhãn rõ `untrained_visobert_throughput_only`).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hieu-kiien/gnn-nlp-propagation.git"
REPO_BRANCH = os.environ.get("SBIF_BRANCH", "main")

is_kaggle = Path("/kaggle/working").exists()
work_root = Path("/kaggle/working/gnn-nlp-propagation") if is_kaggle else Path("/content/gnn-nlp-propagation")

if not Path("scripts/predict_live_thread.py").exists():
    if work_root.exists():
        os.chdir(work_root)
        subprocess.run(["git", "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], check=False)
    else:
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(work_root)], check=True)
        os.chdir(work_root)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)

for mod_name, pip_name in (
    ("torch_geometric", "torch-geometric"),
    ("transformers", "transformers"),
    ("sentencepiece", "sentencepiece"),
    ("google.protobuf", "protobuf"),
    ("huggingface_hub", "huggingface-hub"),
    ("pyarrow", "pyarrow"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("pandas", "pandas"),
    ("lightgbm", "lightgbm"),
    ("statsmodels", "statsmodels"),
    ("pydantic", "pydantic"),
    ("pytest", "pytest"),
    ("requests", "requests"),
):
    try:
        __import__(mod_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

import torch

print("Platform:", "Kaggle" if is_kaggle else "Colab / Linux")
print("Working dir:", Path.cwd())
print("Branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for idx in range(torch.cuda.device_count()):
    print(f"  GPU[{idx}]:", torch.cuda.get_device_name(idx))

In [ ]:
import subprocess
import sys

print("=== 1/3 Running Mega-Thread 10k Scanner unit & integration tests ===")
subprocess.run([sys.executable, "-m", "pytest", "tests/test_megathread_10k_scanner.py", "-v"], check=True)

print("\n=== 2/3 Running full project pytest suite ===")
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

print("\n=== 3/3 Validating 31 published result JSON artifacts ===")
subprocess.run([sys.executable, "-m", "scripts.run_m5_local_audits", "--validate-results"], check=True)

In [ ]:
import json
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer

from scripts.demo_web_app import build_megathread_stress_preset
from scripts.predict_live_thread import predict_live_cascade
from src.nlp.dual_edge_scorer import DualHeadCrossEncoder, VISOBERT_MODEL_NAME

RUN_FULL_10K = True
STRATIFIED_BUDGET = 600
REQUIRED_CE_FILES = ("dual_head_weights.pt", "dual_head_meta.json", "config.json")

# Optional: Mount Google Drive on Colab to pick up MyDrive/SBiFAGAT/checkpoints
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Google Drive mount skipped:", exc)


def _discover_checkpoints() -> tuple[Path | None, Path | None]:
    search_roots = [
        Path("checkpoints"),
        Path("/content/drive/MyDrive/SBiFAGAT/checkpoints"),
        Path("/kaggle/input"),
    ]
    ce_dir: Path | None = None
    gnn_ckpt: Path | None = None
    for root in search_roots:
        if not root.exists():
            continue
        if ce_dir is None:
            direct_ce = root / "dual_ce_vn"
            if direct_ce.is_dir() and all((direct_ce / f).is_file() for f in REQUIRED_CE_FILES):
                ce_dir = direct_ce
            elif root.name == "input":
                for cand in root.rglob("dual_head_weights.pt"):
                    if "vn" in str(cand.parent).lower() and all((cand.parent / f).is_file() for f in REQUIRED_CE_FILES):
                        ce_dir = cand.parent
                        break
        if gnn_ckpt is None:
            direct_gnn = root / "best_sbifagat_vn.pt"
            if direct_gnn.is_file():
                gnn_ckpt = direct_gnn
            elif root.name == "input":
                for cand in root.rglob("best_sbifagat_vn.pt"):
                    gnn_ckpt = cand
                    break
    return ce_dir, gnn_ckpt


ce_dir, gnn_ckpt = _discover_checkpoints()
has_real_ckpts = ce_dir is not None and gnn_ckpt is not None
if has_real_ckpts:
    assert ce_dir is not None
    ce_model, ce_tok = DualHeadCrossEncoder.load_checkpoint(ce_dir, map_location="cpu")
    allow_untrained = False
    weights_mode = "real_trained_checkpoints"
    print(f"Loaded real checkpoints: ce={ce_dir}, gnn={gnn_ckpt}")
else:
    print("Real checkpoints not attached -> initializing ViSoBERT DualHeadCrossEncoder for GPU throughput/VRAM test.")
    ce_tok = AutoTokenizer.from_pretrained(VISOBERT_MODEL_NAME)
    ce_model = DualHeadCrossEncoder(lang="vi", model_name=VISOBERT_MODEL_NAME, edge_dim=2)
    ce_model.eval()
    gnn_ckpt = None
    allow_untrained = True
    weights_mode = "untrained_visobert_throughput_only"

post, comments = build_megathread_stress_preset(10000)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = torch.cuda.get_device_name(0) if gpu_count > 0 else "cpu"


def _run_bench(strategy: str, budget: int) -> dict:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    res = predict_live_cascade(
        post,
        comments,
        k_horizon=20,
        analysis_mode="full",
        max_full_comments=10000,
        scan_strategy=strategy,
        sample_budget=budget,
        preloaded_ce=(ce_model, ce_tok),
        cross_encoder_ckpt=ce_dir,
        gnn_ckpt=gnn_ckpt,
        topic_backend="embedding",
        junk_mode="contract",
        allow_untrained=allow_untrained,
        is_demo=False,
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        peak_mb = max(torch.cuda.max_memory_allocated(i) for i in range(gpu_count)) / (1024 ** 2)
    else:
        peak_mb = 0.0
    elapsed_s = time.perf_counter() - t0
    ms = res.get("mega_scan_summary") or {}
    scored = int(ms.get("scored_edges") or 0)
    return {
        "scan_strategy": strategy,
        "weights_mode": weights_mode,
        "gpu_count": gpu_count,
        "gpu_name": gpu_name,
        "elapsed_sec": round(elapsed_s, 3),
        "scored_edges": scored,
        "edges_per_sec": round(scored / max(elapsed_s, 1e-6), 2),
        "peak_vram_mb": round(peak_mb, 2),
        "kept_comments": ms.get("kept_comments"),
        "coverage_ratio": ms.get("coverage_ratio"),
        "bootstrap_ci_95": ms.get("bootstrap_ci_95"),
        "ood_scale_warning": res.get("ood_scale_warning"),
        "gnn_k_horizon": res.get("gnn_k_horizon"),
        "predicted_total_replies": res.get("predicted_total_replies"),
        "predicted_polarization_score": res.get("predicted_polarization_score"),
        "hotspot_branches_count": len(ms.get("hotspot_branches") or []),
    }


benchmarks = {"stratified_600": _run_bench("stratified", STRATIFIED_BUDGET)}
if RUN_FULL_10K:
    benchmarks["full_10k"] = _run_bench("full", 10000)

out_path = Path("megathread_10k_gpu_benchmark.json")
out_path.write_text(json.dumps(benchmarks, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(benchmarks, ensure_ascii=False, indent=2))
print("Saved benchmark JSON ->", out_path.resolve())

In [ ]:
START_COLAB_WEB_DEMO = False

if START_COLAB_WEB_DEMO:
    import subprocess
    import sys
    import time

    proc = subprocess.Popen(
        [sys.executable, "scripts/demo_web_app.py", "--host", "127.0.0.1", "--port", "8765", "--no-open-browser"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    time.sleep(3)
    try:
        from google.colab import output
        proxy_url = output.eval_js("google.colab.kernel.proxyPort(8765)")
        print("Open Demo Web MVP on Colab ->", proxy_url)
    except Exception as exc:
        print("Local server started on http://127.0.0.1:8765 (proxy helper unavailable:", exc, ")")
else:
    print("Set START_COLAB_WEB_DEMO = True to launch the interactive Mega-Thread 10k Web Demo.")